# Notebook Overview



---


**This notebook demonstrates how to generate, test, and evaluate context from large language models (LLMs) like GPT-4 using relevant contextual information (RAG). It covers the entire workflow starting from data preparation with diverse query-output pairs — including correct, incorrect, harmful, complete, and incomplete responses — through to generating model outputs and evaluating them against multiple KPIs such as:**

- Context Utilization

- Redundancy Reduction

- Relevance Retention


---


The evaluation leverages the LlumoClient API, which provides structured metrics to assess the quality and safety of AI-generated content. Additionally, the notebook includes secure handling of API keys within the Google Colab environment.

In [4]:
# required packages
!pip install openai llumo -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.0/647.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 4.4 MB/s eta 0:00:00


# 🔑 Setup OpenAI API Key & Llumo API key from Colab User Data


In [6]:

# Import the OpenAI client and Colab's userdata module
from openai import OpenAI
from google.colab import userdata

# Retrieve your OpenAI API key from Colab's stored secrets
# ⚠️ Ensure that the required key are saved in Colab using: userdata.set('key_name_here', 'your-api-key-here')
api_key = userdata.get('OPEN_API_KEY')
llumo_key = userdata.get("LLUMO_API_KEY")


# ✅ Sample Data for Evaluating AI Outputs


In [20]:
data = {
    "query": [
        "What is the function of the heart in the human body?",
        "What is the boiling point of water at sea level?",
        "What is the capital of Germany?",
        "Explain the greenhouse effect.",
        "What is quantum computing?",
        "How does photosynthesis help plants survive?",
        "What is the boiling point of mercury?",
        "Who wrote the novel 'Pride and Prejudice'?",
        "Summarize the benefits of electric vehicles based on the given context.",
        "What causes earthquakes?",
        "What are the main functions of the human liver?",
        "Explain the use of wind energy.",
        "What are the effects of global warming?",
        "What causes the seasons on Earth?",
        "What is the water cycle?"



    ],
    "context": [
        # Pass: Fully relevant and coherent
        "The human heart is a muscular organ responsible for pumping blood throughout the body via the circulatory system. It delivers oxygen and nutrients to tissues and removes carbon dioxide and other wastes. The heart beats around 100,000 times a day, supplying oxygenated blood to the organs and muscles while receiving deoxygenated blood back from the veins.",

        # Pass: Accurate and focused
        "At standard atmospheric pressure (1 atmosphere or 101.3 kPa), the boiling point of water is 100 degrees Celsius (212 degrees Fahrenheit). This temperature is affected by altitude and pressure, but at sea level, 100°C is the standard boiling point.",

        # Fail: Ambiguous, contains irrelevant and misleading statements
        "Germany is a European country known for its engineering and beer. While Munich is a cultural hub and Frankfurt is a major financial center, the capital is often mistaken to be Hamburg due to its size. However, some believe Cologne has a richer political history than others.",

        # Fail: Vague, incoherent, lacks clarity
        "The greenhouse thing happens when stuff in the air traps sunlight or heat or something like that. It makes the planet warm, but sometimes it's good, and sometimes it's not. Plants might be involved too, and clouds maybe. Anyway, that’s the idea.",

        # Fail: Scattered, partially relevant with unrelated content
        "Quantum computing is related to quantum mechanics, which also includes things like wave functions and the uncertainty principle. Speaking of uncertainty, weather prediction has gotten better over the years, especially with AI. In ancient Greece, philosophers like Aristotle believed in the geocentric model. Also, bees communicate using dances. Quantum bits or qubits are central to quantum computing.",
                # Full context about photosynthesis
        "Photosynthesis is a process by which green plants and some other organisms use sunlight to synthesize nutrients from carbon dioxide and water. It occurs mainly in the chloroplasts of plant cells. During this process, oxygen is released as a byproduct. This process provides essential glucose for plant energy and growth.",

        # Contains full details about mercury's boiling point
        "Mercury is a dense, silvery liquid metal at room temperature. It has a boiling point of 356.73°C and a freezing point of -38.83°C. Mercury is used in thermometers, barometers, and other scientific instruments due to its unique thermal properties.",

        # No information about Pride and Prejudice
        "The history of English literature includes notable writers such as Charles Dickens, George Orwell, and Virginia Woolf. The Victorian era produced many famous novels and marked a period of strong moral themes in writing.",

        # Rich context on electric vehicles
        "Electric vehicles (EVs) produce zero tailpipe emissions and help reduce greenhouse gases. They are quieter and often cheaper to maintain than gasoline cars. EVs can be charged at home and benefit from renewable energy sources. Government incentives and improving charging infrastructure also promote their adoption.",

        # Weak, scattered context
        "Tectonic plates are large pieces of Earth's crust. Volcanoes erupt due to molten rock. The Richter scale measures magnitude. The earth is made up of different layers, like crust and mantle. Hurricanes are formed over oceans due to warm waters and air pressure.",

        # Good: Clear, no redundancy
        "The human liver processes nutrients from food, produces bile to help digest fats, removes toxins from the blood, and stores energy in the form of glycogen.",

        # Bad: Heavy redundancy (phrasing repetition)
        "Wind energy is generated from the wind. Wind turbines are used to generate wind energy. This energy comes from the movement of wind. Wind energy is renewable. The energy is generated by wind using wind turbines. Wind turbines convert wind into electricity. Wind energy is clean and renewable energy from the wind.",

        # Moderate: Some overlap in concept, not word-for-word
        "Global warming leads to rising temperatures and melting ice caps. Higher temperatures can increase sea levels. It also causes more frequent heatwaves. Melting glaciers due to global warming contribute to sea level rise. Warm temperatures also disrupt ecosystems and weather patterns.",

        # Good: Efficient presentation, no repetition
        "Seasons on Earth occur due to the tilt of Earth's axis. As the Earth orbits the Sun, different parts receive varying amounts of sunlight throughout the year, leading to seasonal changes.",

        # Poor: Restates same concept multiple ways
        "The water cycle describes how water moves through the environment. It includes evaporation, condensation, and precipitation. Water evaporates from surfaces. Then, water condenses into clouds. Precipitation returns water to the surface. The cycle repeats. Water changes form through the water cycle. The water cycle shows how water is recycled."


    ],
    "output": [
        # Good output based on good context
        "The heart pumps blood throughout the body, supplying oxygen and nutrients and removing waste products like carbon dioxide.",

        # Good output based on good context
        "At sea level, the boiling point of water is 100 degrees Celsius or 212 degrees Fahrenheit.",

        # Bad output based on confusing context
        "The capital of Germany is Hamburg, although Cologne and Munich are also important cities.",

        # Bad output based on vague context
        "The greenhouse effect is kind of when heat and air and stuff mix up and make Earth warmer or something like that.",

        # Bad output with minimal relevance
        "Quantum computing involves weather, bees, and ancient Greek theories. It is a mix of science and unpredictability.",

        # Full context used
        "Photosynthesis enables plants to create glucose using sunlight, carbon dioxide, and water. This glucose fuels plant growth, and oxygen is released as a byproduct, making the process vital for survival.",

        # Full context used
        "The boiling point of mercury is 356.73 degrees Celsius.",

        # Context not utilized
        "Pride and Prejudice was written by Charles Dickens in the early 1800s.",

        # Partial context used, missed some points
        "Electric vehicles are better for the environment and help reduce pollution. They are quieter than regular cars.",

        # Context ignored, mixed facts
        "Earthquakes are caused by ocean storms and volcanic eruptions measured by the Beaufort scale.",
        "The liver removes toxins, helps with digestion via bile, and stores energy.",
        "Wind turbines convert wind movement into electricity, offering a clean energy source.",
        "Global warming causes temperature rise, ice melt, and sea-level increase.",
        "Seasons change because of the Earth's tilted axis and orbit around the Sun.",
        "Water moves through stages like evaporation, condensation, and precipitation."


    ]
}


In [21]:
# Necessary imports
import pandas as pd

# Convert the sample data dictionary to a pandas DataFrame
df = pd.DataFrame(data)

# Display the first 5 rows of the DataFrame to verify structure and content
df.head()


,query,context,output
0,What is the function of the heart in the human...,The human heart is a muscular organ responsibl...,"The heart pumps blood throughout the body, sup..."
1,What is the boiling point of water at sea level?,At standard atmospheric pressure (1 atmosphere...,"At sea level, the boiling point of water is 10..."
2,What is the capital of Germany?,Germany is a European country known for its en...,"The capital of Germany is Hamburg, although Co..."
3,Explain the greenhouse effect.,The greenhouse thing happens when stuff in the...,The greenhouse effect is kind of when heat and...
4,What is quantum computing?,Quantum computing is related to quantum mechan...,"Quantum computing involves weather, bees, and ..."


# 🔍 LLM Output Generation using Open AI


In [ ]:
from openai import OpenAI

# Initialize OpenAI client with your API key
client = OpenAI(api_key=api_key)

# List to store model-generated outputs
generated_outputs = []

# Iterate through each row in the DataFrame
for indx, row in df.iterrows():

    # Construct prompt using query and context
    prompt_template = f'Give answer to the given query: {row["query"]}, using the given context: {row["context"]}.'

    # Send the prompt to the OpenAI chat model
    response = client.chat.completions.create(
        model="gpt-4",  # You may also use "gpt-3.5-turbo"
        messages=[
            {"role": "user", "content": prompt_template}
        ],
        temperature=0.7  # Controls randomness in the output
    )

    # Extract the model's reply content from the response
    llm_output = response.choices[0].message.content

    # Append the output to the list
    generated_outputs.append(llm_output)


'Hello Aman, nice to meet you!'

#🔍 LLumo Evaluation: Evaluate Context Using LlumoClient


In [26]:

# Import the evaluation client from Llumo SDK
from llumo import LlumoClient

# Initialize the LlumoClient with your API key
client = LlumoClient(api_key = llumo_key)  # Replace with actual API key

# Evaluate the DataFrame using selected evaluation KPIs
resultdf = client.evaluateMultiple(
    dataframe = df,  # Input DataFrame containing 'query', 'context', and 'output'
    eval = ["Relevance Retention","Context Utilization", "Redundancy Reduction"],  # Selected evaluation KPIs
    prompt_template = "Give answer to the given query: {{query}}, using the given context: {{context}}.",  # Prompt used for generation
    outputColName = "output"  # Column containing model-generated output
)



======= Running evaluation for: Relevance Retention =======

======= Running evaluation for: Context Utilization =======

======= Running evaluation for: Redundancy Reduction =======


🗨 Result DataFrame

In [27]:
resultdf

,query,context,output,Relevance Retention,Relevance Retention Reason,Context Utilization,Context Utilization Reason,Redundancy Reduction,Redundancy Reduction Reason
0,What is the function of the heart in the human...,The human heart is a muscular organ responsibl...,"The heart pumps blood throughout the body, sup...",100,The context provides a concise and accurate ex...,99,The response accurately reflects the core info...,71,The context contains some redundancy. The des...
1,What is the boiling point of water at sea level?,At standard atmospheric pressure (1 atmosphere...,"At sea level, the boiling point of water is 10...",99,The context provides the boiling point of wate...,99,The response accurately reflects the boiling p...,92,The context mostly avoids redundancy. The boi...
2,What is the capital of Germany?,Germany is a European country known for its en...,"The capital of Germany is Hamburg, although Co...",74,The context mentions several German cities but...,56,"The response uses some context, correctly iden...",81,The context contains some redundancy. 'Munich...
3,Explain the greenhouse effect.,The greenhouse thing happens when stuff in the...,The greenhouse effect is kind of when heat and...,74,The context provides a basic explanation of th...,58,The response uses some relevant information fr...,73,The context contains some redundancy. Phrases...
4,What is quantum computing?,Quantum computing is related to quantum mechan...,"Quantum computing involves weather, bees, and ...",71,The context provides some relevant information...,19,The response fabricates information not presen...,78,The context contains some redundancy. 'Quantu...
5,How does photosynthesis help plants survive?,Photosynthesis is a process by which green pla...,Photosynthesis enables plants to create glucos...,99,The context provides a concise and accurate ex...,99,The response accurately reflects the context b...,85,The context mostly avoids repetition. However...
6,What is the boiling point of mercury?,"Mercury is a dense, silvery liquid metal at ro...",The boiling point of mercury is 356.73 degrees...,100,The context directly provides the boiling poin...,99,The response accurately extracts the boiling p...,99,The context presents information concisely. T...
7,Who wrote the novel 'Pride and Prejudice'?,The history of English literature includes not...,Pride and Prejudice was written by Charles Dic...,1,The provided context lacks the information nee...,14,The response incorrectly attributes *Pride and...,100,"The context presents information concisely, av..."
8,Summarize the benefits of electric vehicles ba...,Electric vehicles (EVs) produce zero tailpipe ...,Electric vehicles are better for the environme...,100,The context provides a concise summary of EV b...,81,The response uses some relevant information fr...,90,"The context presents information concisely, wi..."
9,What causes earthquakes?,Tectonic plates are large pieces of Earth's cr...,Earthquakes are caused by ocean storms and vol...,1,The context lacks information about earthquake...,15,The response incorrectly states that earthquak...,93,The context presents distinct geological conce...
